# Day 3: Microstructure Modeling (NODDI & DKI)

Welcome to the **Day 3 Interactive Microstructure Laboratory**.

In this notebook, we interactively explore:
1. **Neurite Density Index (NDI / ICVF)**: Intra-cellular volume fraction (tight axonal packing in major white matter tracts).
2. **Orientation Dispersion Index (ODI)**: Angular dispersion of neurites (high in crossing/fanning regions, low in parallel bundles like the Corpus Callosum).
3. **Mean Kurtosis (MK)**: Deviation from Gaussian diffusion sensitive to microstructural complexity and cellular packing.

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
%matplotlib inline

def find_file(filename):
    candidates = [
        filename,
        os.path.join("..", filename),
        os.path.join("NODDI_results", "FIT_ICVF.nii.gz") if "NDI" in filename else filename,
        os.path.join("..", "NODDI_results", "FIT_ICVF.nii.gz") if "NDI" in filename else filename
    ]
    for path in candidates:
        if os.path.exists(path):
            return path
    return None

ndi_path = find_file("NDI.nii.gz")
odi_path = find_file("ODI.nii.gz") or find_file("NODDI_results/FIT_OD.nii.gz")
mk_path = find_file("dki_mk.nii.gz")
fa_path = find_file("fa.nii.gz")

print(f"NDI file : {ndi_path}")
print(f"ODI file : {odi_path}")
print(f"MK file  : {mk_path}")
print(f"FA file  : {fa_path}")

In [ ]:
if ndi_path and os.path.exists(ndi_path):
    ndi_data = nib.load(ndi_path).get_fdata()
    odi_data = nib.load(odi_path).get_fdata() if odi_path and os.path.exists(odi_path) else None
    mk_data = nib.load(mk_path).get_fdata() if mk_path and os.path.exists(mk_path) else None
    fa_data = nib.load(fa_path).get_fdata() if fa_path and os.path.exists(fa_path) else None

    slice_z = ndi_data.shape[2] // 2

    n_plots = 1 + (odi_data is not None) + (mk_data is not None) + (fa_data is not None)
    fig, axes = plt.subplots(1, n_plots, figsize=(4 * n_plots, 4))
    if n_plots == 1:
        axes = [axes]
    
    idx = 0
    # 1. FA
    if fa_data is not None:
        im0 = axes[idx].imshow(np.rot90(fa_data[:, :, slice_z]), cmap="gray", vmin=0, vmax=0.9)
        axes[idx].set_title(f"DTI FA (Slice {slice_z})")
        axes[idx].axis("off")
        plt.colorbar(im0, ax=axes[idx], fraction=0.046, pad=0.04)
        idx += 1
    
    # 2. NDI
    im1 = axes[idx].imshow(np.rot90(ndi_data[:, :, slice_z]), cmap="plasma", vmin=0, vmax=1.0)
    axes[idx].set_title(f"NODDI NDI (Slice {slice_z})")
    axes[idx].axis("off")
    plt.colorbar(im1, ax=axes[idx], fraction=0.046, pad=0.04)
    idx += 1

    # 3. ODI
    if odi_data is not None:
        im2 = axes[idx].imshow(np.rot90(odi_data[:, :, slice_z]), cmap="viridis", vmin=0, vmax=0.8)
        axes[idx].set_title(f"NODDI ODI (Slice {slice_z})")
        axes[idx].axis("off")
        plt.colorbar(im2, ax=axes[idx], fraction=0.046, pad=0.04)
        idx += 1

    # 4. MK
    if mk_data is not None:
        im3 = axes[idx].imshow(np.rot90(mk_data[:, :, slice_z]), cmap="magma", vmin=0, vmax=2.0)
        axes[idx].set_title(f"DKI MK (Slice {slice_z})")
        axes[idx].axis("off")
        plt.colorbar(im3, ax=axes[idx], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.show()
else:
    print("Microstructure parameter maps not found yet. Please run tutorial_3.3_microstructure.sh first.")